# Google Colab notebook for EMIT-L2A product download

Based on EMIT-L2B $CH_4$ product, this notebook retrieves the corresponding L2A acquisitions and downloads the hyperspectral data cube to pair it with the plume annotation.

### Authenticate to the GEE project and connect to Drive

In [1]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import re
from glob import glob
import rasterio as rio
from google.colab import drive
from shapely.geometry import box

ee.Authenticate()
ee.Initialize(project='lithe-timer-424608-c4')

drive.mount("/content/drive/")

Mounted at /content/drive/


### CONFIGS

In [ ]:
%cd '/content/drive/MyDrive/EMIT_CH4/'

# Path to both the CSV and GPKG file
CSV_FILE = './emit_ch4_plumes_metadata.csv'
GPKG_FILE = './emit_ch4_plumes_metadata.gpkg'

df = pd.read_csv(CSV_FILE)
gdf = gpd.read_file(GPKG_FILE)

/content/drive/MyDrive/EMIT_CH4


### PoC on one plume

In [ ]:
EXP_ID = 88
exp_plume = gdf.iloc[EXP_ID]

single_plume_gdf = gpd.GeoDataFrame([exp_plume], geometry='geometry', crs="EPSG:4326")
utm_crs = single_plume_gdf.estimate_utm_crs()
plume_utm = single_plume_gdf.to_crs(utm_crs)

In [7]:
centroid_utm = plume_utm.geometry.centroid.iloc[0]
cx_m, cy_m = centroid_utm.x, centroid_utm.y
radius_m = 15360
minx, miny = cx_m - radius_m, cy_m - radius_m
maxx, maxy = cx_m + radius_m, cy_m + radius_m
bbox_utm = gpd.GeoSeries([box(minx, miny, maxx, maxy)], crs=utm_crs)
bbox_latlon = bbox_utm.to_crs("EPSG:4326").iloc[0]
ee_bbox = ee.Geometry.Polygon(list(bbox_latlon.exterior.coords))
roi_point = ee.Geometry.Point([exp_plume.geometry.centroid.x, exp_plume.geometry.centroid.y])

In [8]:
center_date = ee.Date.parse('YYYY-MM-dd\'T\'HH:mm:ss', exp_plume.startDate[:-1])
start_time = center_date.advance(-5, 'minute')
end_time = center_date.advance(5, 'minute')

In [9]:
idx = [i for i in range(286)]

In [10]:
collection = ee.ImageCollection("NASA/EMIT/L2A/RFL") \
        .filterDate(start_time, end_time) \
        .filterBounds(ee_bbox) \
        .select(idx)

In [11]:
collection.size().getInfo()

1

In [12]:
ee_crs = f"EPSG:{utm_crs.to_epsg()}"

In [13]:
image_mosaic = collection.mosaic().unmask(0)

In [14]:
roi_geometry = ee.Geometry(exp_plume.geometry.__geo_interface__)
mask_mosaic = ee.Image.constant(0).byte() \
    .paint(roi_geometry, 1) \
    .unmask(0)

In [15]:
image_512 = image_mosaic.reproject(crs=ee_crs, scale=60).clip(ee_bbox)
mask_512 = mask_mosaic.reproject(crs=ee_crs, scale=60).clip(ee_bbox)

In [16]:
image_512.bandNames().getInfo()

['reflectance_0',
 'reflectance_1',
 'reflectance_2',
 'reflectance_3',
 'reflectance_4',
 'reflectance_5',
 'reflectance_6',
 'reflectance_7',
 'reflectance_8',
 'reflectance_9',
 'reflectance_10',
 'reflectance_11',
 'reflectance_12',
 'reflectance_13',
 'reflectance_14',
 'reflectance_15',
 'reflectance_16',
 'reflectance_17',
 'reflectance_18',
 'reflectance_19',
 'reflectance_20',
 'reflectance_21',
 'reflectance_22',
 'reflectance_23',
 'reflectance_24',
 'reflectance_25',
 'reflectance_26',
 'reflectance_27',
 'reflectance_28',
 'reflectance_29',
 'reflectance_30',
 'reflectance_31',
 'reflectance_32',
 'reflectance_33',
 'reflectance_34',
 'reflectance_35',
 'reflectance_36',
 'reflectance_37',
 'reflectance_38',
 'reflectance_39',
 'reflectance_40',
 'reflectance_41',
 'reflectance_42',
 'reflectance_43',
 'reflectance_44',
 'reflectance_45',
 'reflectance_46',
 'reflectance_47',
 'reflectance_48',
 'reflectance_49',
 'reflectance_50',
 'reflectance_51',
 'reflectance_52',
 'r

In [17]:
image_vis = {
    'bands': ['reflectance_36', 'reflectance_23', 'reflectance_12'],
    'min' : 0,
    'max' : 0.3,
}

mask_vis = {
    'min': 0,
    'max': 1,
    'palette': ['black', 'red']
}

map = geemap.Map(center=[exp_plume.geometry.centroid.y, exp_plume.geometry.centroid.x], zoom=11)
map.addLayer(image_512, image_vis, 'EMIT 512x512')
map.addLayer(mask_512, mask_vis, 'Plume Mask', opacity=0.5)

display(map)

TimeoutException: Requesting secret GOOGLE_MAPS_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

## Expanding PoC to the full dataset

### Create the BBOXes used to crop the EMIT satellite image

In [ ]:
centroids = gdf.geometry.centroid
lon = centroids.x
lat = centroids.y
utm_zone = np.floor((lon + 180) / 6) + 1
epsg_base = np.where(lat >= 0, 32600, 32700)
gdf['utm_epsg'] = "EPSG:" + (epsg_base + utm_zone).astype(int).astype(str)
gee_bboxes = gpd.GeoSeries(index=gdf.index, crs="EPSG:4326")

for epsg, group in gdf.groupby('utm_epsg'):
  group_utm = group.to_crs(epsg)
  is_small = (group['width_km'] < 25) & (group['height_km'] < 25)
  small_utm_bboxes = group_utm[is_small].geometry.centroid.buffer(15360, cap_style=3)
  large_utm_bboxes = group_utm[~is_small].geometry.envelope.buffer(500, cap_style=3)
  combined_utm_bboxes = pd.concat([small_utm_bboxes, large_utm_bboxes])
  latlon_squares = gpd.GeoSeries(combined_utm_bboxes, crs=epsg).to_crs("EPSG:4326")
  gee_bboxes.loc[group.index] = latlon_squares

gdf['gee_bbox'] = gee_bboxes
gdf['is_fixed_512'] = (gdf['width_km'] < 25) & (gdf['height_km'] < 25)

/tmp/ipython-input-956450794.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


### Download script

In [ ]:
from pathlib import Path
import concurrent.futures
IMG_DIR = Path('./EMIT_L2A_chips')
MASK_DIR = Path('./CH4_plumes_masks')
#IMG_DIR.mkdir(exist_ok=True)
#MASK_DIR.mkdir(exist_ok=True)

In [ ]:
gdf_batch1 = gdf.iloc[:500]
gdf_batch2 = gdf.iloc[500:1000]
gdf_batch3 = gdf.iloc[1000:]

In [ ]:
len(gdf_batch3)

574

In [ ]:
bands = [i for i in range(285)]
for idx, plume in gdf_batch3.iterrows():

  plume_id = plume['name']
  date_str = plume['name'][:15]

  img_out = IMG_DIR / f"{plume_id}.tif"
  mask_out = MASK_DIR / f"{plume_id}.tif"

  coords = list(plume.gee_bbox.exterior.coords)
  ee_bbox = ee.Geometry.Polygon(coords)

  ee_polygon = ee.Geometry(plume.geometry.__geo_interface__)

  center_date = ee.Date.parse('YYYYMMdd\'T\'HHmmss', date_str)

  collection = ee.ImageCollection("NASA/EMIT/L2A/RFL") \
        .filterDate(center_date.advance(-5, 'minute'), center_date.advance(5, 'minute')) \
        .filterBounds(ee_bbox) \
        .select(bands)

  image = collection.mosaic() \
            .unmask(0) \
            .reproject(crs=plume['utm_epsg'], scale=60) \
            .clip(ee_bbox)

  mask = ee.Image.constant(0).byte() \
            .paint(ee_polygon, 1) \
            .unmask(0) \
            .reproject(crs=plume['utm_epsg'], scale=60) \
            .clip(ee_bbox)

  try:
    task_img = ee.batch.Export.image.toDrive(
            image=image,
            description=f"IMG_{plume_id}", # Task name in GEE
            folder="EMIT_L2A_chips",                   # Folder created in your Google Drive
            fileNamePrefix=plume_id, # Actual file name
            region=ee_bbox,
            scale=60,
            crs=plume['utm_epsg'],
            maxPixels=1e10 # Prevents arbitrary pixel limit errors
        )
    task_img.start() # Submits to Google's servers and moves on instantly

    task_mask = ee.batch.Export.image.toDrive(
            image=mask,
            description=f"MASK_{plume_id}",
            folder="CH4_plumes_masks",
            fileNamePrefix=plume_id,
            region=ee_bbox,
            scale=60,
            crs=plume['utm_epsg'],
            maxPixels=1e10
        )
    task_mask.start()

    print(f"Submitted {plume_id} to GEE Batch queue.")

  except Exception as e:
      print(f"Failed to submit {plume_id}: {e}")

In [ ]:
import time
from collections import Counter
from IPython.display import clear_output

try:
    while True:
        tasks = ee.batch.Task.list()[:1148]
        status_counts = Counter([task.state for task in tasks])

        clear_output(wait=True)

        print("LIVE GEE Task Monitor")
        print(f"Last updated: {time.strftime('%H:%M:%S')}\n")

        for state, count in status_counts.items():
            print(f"{state:10}: {count} tasks")

        if status_counts.get('READY', 0) == 0 and status_counts.get('RUNNING', 0) == 0:
            print("\nAll tasks have finished processing!")
            break

        time.sleep(60)

except KeyboardInterrupt:
    print("\nLive monitor stopped by user.")

LIVE GEE Task Monitor
Last updated: 14:25:47

READY     : 327 tasks
RUNNING   : 2 tasks
COMPLETED : 819 tasks
LIVE GEE Task Monitor
Last updated: 14:26:49

READY     : 327 tasks
RUNNING   : 2 tasks
COMPLETED : 819 tasks
LIVE GEE Task Monitor
Last updated: 14:27:51

READY     : 327 tasks
RUNNING   : 2 tasks
COMPLETED : 819 tasks
LIVE GEE Task Monitor
Last updated: 14:28:53

READY     : 327 tasks
RUNNING   : 2 tasks
COMPLETED : 819 tasks
LIVE GEE Task Monitor
Last updated: 14:29:55

READY     : 327 tasks
RUNNING   : 2 tasks
COMPLETED : 819 tasks
LIVE GEE Task Monitor
Last updated: 14:30:57

READY     : 326 tasks
RUNNING   : 2 tasks
COMPLETED : 820 tasks
LIVE GEE Task Monitor
Last updated: 14:31:59

READY     : 325 tasks
RUNNING   : 2 tasks
COMPLETED : 821 tasks
LIVE GEE Task Monitor
Last updated: 14:33:01

READY     : 323 tasks
RUNNING   : 2 tasks
COMPLETED : 823 tasks
LIVE GEE Task Monitor
Last updated: 14:34:03

READY     : 323 tasks
RUNNING   : 2 tasks
COMPLETED : 823 tasks
LIVE GEE T

LIVE GEE Task Monitor
Last updated: 15:05:04

READY     : 311 tasks
RUNNING   : 2 tasks
COMPLETED : 835 tasks
LIVE GEE Task Monitor
Last updated: 15:06:06

READY     : 311 tasks
RUNNING   : 2 tasks
COMPLETED : 835 tasks
LIVE GEE Task Monitor
Last updated: 15:07:08

READY     : 311 tasks
RUNNING   : 2 tasks
COMPLETED : 835 tasks
LIVE GEE Task Monitor
Last updated: 15:08:10

READY     : 309 tasks
RUNNING   : 2 tasks
COMPLETED : 837 tasks
LIVE GEE Task Monitor
Last updated: 15:09:12

READY     : 309 tasks
RUNNING   : 2 tasks
COMPLETED : 837 tasks
LIVE GEE Task Monitor
Last updated: 15:10:14

READY     : 308 tasks
RUNNING   : 2 tasks
COMPLETED : 838 tasks
LIVE GEE Task Monitor
Last updated: 15:11:16

READY     : 308 tasks
RUNNING   : 2 tasks
COMPLETED : 838 tasks
LIVE GEE Task Monitor
Last updated: 15:12:19

READY     : 307 tasks
RUNNING   : 2 tasks
COMPLETED : 839 tasks
LIVE GEE Task Monitor
Last updated: 15:13:21

READY     : 305 tasks
RUNNING   : 2 tasks
COMPLETED : 841 tasks
LIVE GEE T

LIVE GEE Task Monitor
Last updated: 16:01:58

READY     : 285 tasks
RUNNING   : 2 tasks
COMPLETED : 861 tasks
LIVE GEE Task Monitor
Last updated: 16:03:00

READY     : 285 tasks
RUNNING   : 2 tasks
COMPLETED : 861 tasks
LIVE GEE Task Monitor
Last updated: 16:04:03

READY     : 285 tasks
RUNNING   : 2 tasks
COMPLETED : 861 tasks
LIVE GEE Task Monitor
Last updated: 16:05:06

READY     : 285 tasks
RUNNING   : 2 tasks
COMPLETED : 861 tasks
LIVE GEE Task Monitor
Last updated: 16:06:08

READY     : 283 tasks
RUNNING   : 2 tasks
COMPLETED : 863 tasks
LIVE GEE Task Monitor
Last updated: 16:07:10

READY     : 282 tasks
RUNNING   : 3 tasks
COMPLETED : 863 tasks
LIVE GEE Task Monitor
Last updated: 16:08:12

READY     : 282 tasks
COMPLETED : 864 tasks
RUNNING   : 2 tasks
LIVE GEE Task Monitor
Last updated: 16:09:14

READY     : 281 tasks
RUNNING   : 2 tasks
COMPLETED : 865 tasks
LIVE GEE Task Monitor
Last updated: 16:10:16

READY     : 281 tasks
RUNNING   : 2 tasks
COMPLETED : 865 tasks
LIVE GEE T

LIVE GEE Task Monitor
Last updated: 16:58:53

READY     : 261 tasks
RUNNING   : 2 tasks
COMPLETED : 885 tasks
LIVE GEE Task Monitor
Last updated: 16:59:55

READY     : 261 tasks
RUNNING   : 2 tasks
COMPLETED : 885 tasks
LIVE GEE Task Monitor
Last updated: 17:00:57

READY     : 261 tasks
RUNNING   : 2 tasks
COMPLETED : 885 tasks
LIVE GEE Task Monitor
Last updated: 17:01:59

READY     : 261 tasks
RUNNING   : 2 tasks
COMPLETED : 885 tasks
LIVE GEE Task Monitor
Last updated: 17:03:02

READY     : 261 tasks
RUNNING   : 2 tasks
COMPLETED : 885 tasks
LIVE GEE Task Monitor
Last updated: 17:04:04

READY     : 258 tasks
RUNNING   : 2 tasks
COMPLETED : 888 tasks
LIVE GEE Task Monitor
Last updated: 17:05:06

READY     : 257 tasks
RUNNING   : 2 tasks
COMPLETED : 889 tasks
LIVE GEE Task Monitor
Last updated: 17:06:09

READY     : 257 tasks
RUNNING   : 2 tasks
COMPLETED : 889 tasks
LIVE GEE Task Monitor
Last updated: 17:07:11

READY     : 257 tasks
RUNNING   : 2 tasks
COMPLETED : 889 tasks
LIVE GEE T

LIVE GEE Task Monitor
Last updated: 17:55:44

READY     : 235 tasks
RUNNING   : 2 tasks
COMPLETED : 911 tasks
LIVE GEE Task Monitor
Last updated: 17:56:46

READY     : 234 tasks
RUNNING   : 2 tasks
COMPLETED : 912 tasks
LIVE GEE Task Monitor
Last updated: 17:57:49

READY     : 233 tasks
RUNNING   : 2 tasks
COMPLETED : 913 tasks
LIVE GEE Task Monitor
Last updated: 17:58:51

READY     : 233 tasks
RUNNING   : 2 tasks
COMPLETED : 913 tasks
LIVE GEE Task Monitor
Last updated: 17:59:53

READY     : 233 tasks
RUNNING   : 2 tasks
COMPLETED : 913 tasks
LIVE GEE Task Monitor
Last updated: 18:00:56

READY     : 233 tasks
RUNNING   : 2 tasks
COMPLETED : 913 tasks
LIVE GEE Task Monitor
Last updated: 18:01:58

READY     : 233 tasks
RUNNING   : 2 tasks
COMPLETED : 913 tasks
LIVE GEE Task Monitor
Last updated: 18:03:00

READY     : 232 tasks
RUNNING   : 2 tasks
COMPLETED : 914 tasks
LIVE GEE Task Monitor
Last updated: 18:04:02

READY     : 231 tasks
RUNNING   : 2 tasks
COMPLETED : 915 tasks
LIVE GEE T

LIVE GEE Task Monitor
Last updated: 18:52:38

READY     : 213 tasks
RUNNING   : 2 tasks
COMPLETED : 933 tasks
LIVE GEE Task Monitor
Last updated: 18:53:40

READY     : 212 tasks
COMPLETED : 935 tasks
RUNNING   : 1 tasks
LIVE GEE Task Monitor
Last updated: 18:54:42

READY     : 211 tasks
RUNNING   : 2 tasks
COMPLETED : 935 tasks
LIVE GEE Task Monitor
Last updated: 18:55:44

READY     : 211 tasks
RUNNING   : 2 tasks
COMPLETED : 935 tasks


### Checking the test run

In [ ]:
download_img_path = "./TEST_EMIT_Raw_Images"
download_mask_path = "./TEST_EMIT_Raw_Masks"
images = glob(f"{download_img_path}/*.tif")
images

['./EMIT_Raw_Images/20220810T064957_000485.tif',
 './EMIT_Raw_Images/20220810T065021_000486.tif',
 './EMIT_Raw_Images/20220810T065132_000496.tif',
 './EMIT_Raw_Images/20220810T065132_000487.tif',
 './EMIT_Raw_Images/20220811T042630_000490.tif']

In [ ]:
dataset = rio.open(images[2])
dataset.name


'./EMIT_Raw_Images/20220810T065132_000496.tif'